In [1]:
# 0. Environment Setup

# Clone the gemma repository
!git clone https://github.com/google-deepmind/gemma.git || true
!pip install -q ./gemma

# Clone the dialog repository for UI/UX
!git clone https://github.com/google-deepmind/dialog.git || true
!pip install -q ./dialog

# Ensure local modules are in path
import sys
import os
sys.path.append(os.getcwd())

Cloning into 'gemma'...
remote: Enumerating objects: 2488, done.
remote: Counting objects: 100% (1139/1139), done.
remote: Compressing objects: 100% (362/362), done.
remote: Total 2488 (delta 927), reused 778 (delta 777), pack-reused 1349 (from 3)
Receiving objects: 100% (2488/2488), 1.19 MiB | 3.76 MiB/s, done.
Resolving deltas: 100% (1754/1754), done.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 71.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 318.4/318.4 kB 33.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 633.8/633.8 kB 53.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 619.6/619.6 kB 53.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.1/204.1 kB 24.9 MB/s

In [2]:
!git clone https://github.com/andrew-veriga/Titans_jax.git
%cd Titans_jax

Cloning into 'Titans_jax'...
remote: Enumerating objects: 327, done.
remote: Counting objects: 100% (72/72), done.
remote: Compressing objects: 100% (22/22), done.
remote: Total 327 (delta 59), reused 59 (delta 50), pack-reused 255 (from 1)
Receiving objects: 100% (327/327), 28.94 MiB | 20.61 MiB/s, done.
Resolving deltas: 100% (193/193), done.
/content/Titans_jax


In [3]:
import importlib
import jax
import os
import gemma_titans
# importlib.reload(gemma_titans)
from gemma_titans import Gemma3_1B_Titans, Gemma_Titans_Config
from titans_ckpts import SkipTitans
import titans_tree_utils
from gemma import gm

print(f"JAX Backend: {jax.default_backend()}")
print(f"Devices: {jax.devices()}")

os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = ".85"

# jax.config.update("jax_disable_jit", True) # Temporarily disable JIT to bypass the hashing error

JAX Backend: gpu
Devices: [CudaDevice(id=0)]


In [4]:
import google.colab
import shutil
import orbax.checkpoint as ocp

if os.path.exists('/content/Titans_jax/saved_titans_delta.zip'):
    shutil.unpack_archive('/content/Titans_jax/saved_titans_delta.zip',"./saved_titans_delta")

def load_titans_weights(load_dir: str):
    load_path = os.path.abspath(load_dir)
    checkpointer = ocp.StandardCheckpointer()
    return checkpointer.restore(load_path)

import jax.numpy as jnp
gemma_config = Gemma3_1B_Titans.config

model = Gemma3_1B_Titans(
    config=gemma_config,
    dtype=jnp.bfloat16,
    return_last_only=False,
    tokens="batch.input",
)
print("Loading weights...")
original_params = gm.ckpts.load_params(gm.ckpts.CheckpointPath.GEMMA3_1B_IT)
loaded_titans_params = load_titans_weights("./saved_titans_delta")
merged_params = titans_tree_utils.merge_titans_params(original_params, loaded_titans_params)

tokenizer = gm.text.Gemma3Tokenizer()

Loading weights...


## 4. Interactive Dialogue with `google-deepmind/dialog`

In [5]:
import dialog
from gemma import gm
import jax

# Initialize Sampler and Conversation
sampler = gm.text.Sampler(
    model=model,
    params=merged_params,
    tokenizer=tokenizer,
)

conv = dialog.Conversation()


def chat(user_input: str):
    global conv
    # Add user message
    conv += dialog.User(user_input)

    # Convert conversation to prompt (Gemma 3 format)
    prompt = conv.as_text(training=False)

    # Generate response
    # Note: Sampler handles the caching of Titans memory automatically
    response_text = sampler.sample(prompt, max_new_tokens=128)

    # Add model response to UI
    conv += dialog.Model(response_text)
    conv.show()

# Example usage:
# chat("Привет! Кто такие Титаны в мифологии?")

In [6]:
chat("Привет! Кто такие Титаны в мифологии?")
